# Човек-в-цикъла: Предварителни пропуски, Категоризиране на риска и Одитно логване

README файлът за този урок въвежда Човек-в-цикъла с кратък откъс, който пита потребителя дали да `ОДОБРИ` или `ОТКАЖЕ`, след като агентът вече е генерирал отговор. Този модел е добър начален подход, но производствените реализации на HITL обикновено изискват три допълнителни елемента:

1. **предварителен пропуск** който се изпълнява **преди** агентът да предприеме рискована стъпка, така че разходите, необратимостта и забавянето да бъдат под контрол.
2. **категоризиране на риска**, така че действия с нисък риск да се изпълняват автоматично, действия със среден риск да се одобряват на партиди, и само действия с висок риск да изискват намесата на човек.
3. **одитен журнал плюс цикъл за ревизия**, така че всяко решение за пропускане да се записва като JSONL, а отхвърлянето да подканва агента с структуриран мотив вместо просто да извежда `Преразглеждане...`.

Този ноутбук изгражда всеки от тези елементи върху същите примитиви като в `06-system-message-framework.ipynb`. Стартира се изцяло в режим `DEMO_MODE = True` (без нужда от интерактивен вход) или с реални `input()` подсказки, когато `DEMO_MODE = False`. Забележка: в DEMO_MODE повторното опитване по третата цел е скриптирано, за да се вижда механиката на цикъла изцяло. Истинското преразглеждане и повторна класификация изисква `DEMO_MODE = False` и оператор.

**Извън обсега (обработва се в други уроци):** удостоверяване и контрол на достъпа (заплаха 2 от README в Урок 06), междинен софтуер за извикване на инструменти (дълбок анализ в Урок 14 MAF), модели на мултиагентен дебат.


In [ ]:
import json
import os
from datetime import datetime, timezone
from pathlib import Path

from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from openai import OpenAI

load_dotenv()

DEMO_MODE = True  # set False to use real input() prompts

# Per-run unique log filename so demo runs don't overwrite each other and
# the notebook doesn't touch any pre-existing gate_log.jsonl in the working
# directory.
GATE_LOG_PATH = Path(
    f"gate_log_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}.jsonl"
)

# This notebook uses the Azure OpenAI Responses API via the stable /openai/v1/ endpoint.
# GitHub Models is deprecated (retiring July 2026) and does not support the Responses API.
endpoint = os.environ.get("AZURE_OPENAI_ENDPOINT", "")
if not endpoint:
    raise RuntimeError(
        "AZURE_OPENAI_ENDPOINT environment variable is not set. This notebook needs "
        "an Azure OpenAI resource with a model deployment that supports the Responses "
        "API. Set AZURE_OPENAI_ENDPOINT and AZURE_OPENAI_DEPLOYMENT in "
        "your environment or a local .env file, then run `az login`."
    )

deployment = os.environ["AZURE_OPENAI_DEPLOYMENT"]

# Authenticate with Entra ID (run `az login` first). No api_version is needed.
token_provider = get_bearer_token_provider(
    DefaultAzureCredential(),
    "https://cognitiveservices.azure.com/.default",
)

client = OpenAI(
    base_url=f"{endpoint.rstrip('/')}/openai/v1/",
    api_key=token_provider,
)



## Модел 1: Предварително действие врата

Извадката HITL в README първо извиква агента, а след това пита потребителя за одобрение на резултата. Това е поток с **пост-действие**. Агентът вече е изпълнил, така че LLM разходът вече е платен, и всякакви странични ефекти (изпратен имейл, записан ред в база данни, публикуван коментар) вече са се случили.

Един поток с **предварително действие** поставя вратата преди агентът да изпълни рисковата стъпка. Агентът предлага действието, вратата решава дали да се изпълни, и само при одобрение настъпва страничният ефект.

| Аспект | Одобрение след действие (извадка от README) | Врата преди действие (този тефтер) |
|---|---|---|
| Кога се изпълнява одобрението? | След като агентът е произвел резултат | Преди да се изпълни какъвто и да е страничен ефект |
| Разход за LLM при отказ | Вече платен | Плаща се само за предложението, не за действието |
| Необратими странични ефекти | Възможни (действието вече се е случило) | Предотвратени |
| Яснота на одита | Одобрението е печатна команда | Одобрението е JSON запис с времева маркировка, действие, причина |


In [ ]:
def gate_action(action_description: str, risk_tier: str, attempt: int = 0) -> dict:
    """Run a single pre-action gate.

    Returns a decision dict with keys: decision, reason, ts.
    Decision is one of: approve, deny, escalate.
    Safe default on EOF or unexpected input is deny.

    DEMO_MODE behavior: high-risk actions are denied on attempt 0 and
    auto-approved on attempt >= 1. This is scripted approval to show the
    loop mechanics (deny -> retry -> approve). It is NOT revision-driven
    re-classification. Real revision-driven re-classification requires
    DEMO_MODE=False and a human operator who evaluates the revised
    proposal on its own merits.
    """
    print(f"[gate] proposed action ({risk_tier}, attempt={attempt}): {action_description}")

    if DEMO_MODE:
        if risk_tier == "high":
            decision = "approve" if attempt >= 1 else "deny"
            reason = (
                "DEMO_MODE: scripted approval on retry to show loop mechanics"
                if attempt >= 1
                else "DEMO_MODE: high risk denied on first attempt"
            )
        else:
            decision = "approve"
            reason = f"DEMO_MODE canned response for tier={risk_tier}"
    else:
        try:
            raw = input("[gate] approve / deny / escalate? ").strip().lower()
        except EOFError:
            raw = ""
        if raw in {"approve", "deny", "escalate"}:
            decision, reason = raw, "operator input"
        elif raw == "":
            decision, reason = "deny", "no input received, defaulted to deny"
        else:
            decision, reason = "deny", f"invalid input {raw!r}, defaulted to deny"

    return {
        "decision": decision,
        "reason": reason,
        "action": action_description,
        "risk_tier": risk_tier,
        "ts": datetime.now(timezone.utc).isoformat(),
    }


## Модел 2: Рискови нива

Не всяко действие изисква одобрение от човек. Просто търсене само за четене чрез публично API има различни рискове в сравнение с изпращане на имейл на клиент. Третирането им по един и същ начин разсейва оператора и забавя агента.

Прост модел с 3 нива:

| Ниво | Примери | Поток на одобрение |
|---|---|---|
| `нисък` (само за четене) | Търсене в база знания, проверка на опции за полет, извличане на публична уеб страница | Автоматично изпълнение, логнато за одит |
| `среден` (евтина мутация) | Кеширане на резултат, увеличаване на брояча, задаване на напомняне | Автоматично изпълнение, но с групов дневен преглед |
| `висок` (външен или необратим) | Изпращане на имейл, таксуване на карта, публикуване в публичен канал | Блокиране до одобрение от човек |

Това е едно ниворане. Продукционните системи често използват по-фини нива (например, нива на разрешения в AWS IAM, нива на достъп на база роли). Тристепенната версия по-долу е най-малката полезна версия за агент, който смесва действия само за четене и такива със странични ефекти.

Класификаторът по-долу използва евристики с ключови думи, за да остане демото детерминистично и евтино. В продукционна система бихте заменили това с изкуствен класификатор или модул за политики.


In [ ]:
LOW_RISK_KEYWORDS = {
    "look", "lookup", "search", "fetch", "read", "query", "view",
    "get", "list", "weather", "summarize",
}
HIGH_RISK_KEYWORDS = {
    "send", "email", "post", "publish", "charge", "pay", "transfer",
    "delete", "drop", "cancel", "refund",
}
MEDIUM_RISK_KEYWORDS = {
    "cache", "schedule", "reminder", "book", "reserve", "update",
    "increment", "log",
}

AUTO_APPROVE_REASONS = {
    "low": "auto-approved (low risk)",
    "medium": "auto-approved (medium risk, queued for batched review)",
}


def classify_risk(action: str) -> str:
    """Classify an action string into one of: low, medium, high.

    Keyword-based heuristic. Checks high-risk first (most severe), then
    low-risk explicit reads, then medium-risk mutations. Unrecognized
    actions default to medium, not low.

    Default for unrecognized actions is 'medium', not 'low'. A read-only
    keyword set will always have blind spots, and the parent README's
    threat list (critical-system access, knowledge-base poisoning,
    cascading errors) all involve cases an action-name alone cannot rule
    out. Routing unknown actions through batched review is the safer
    default than auto-executing them.
    """
    text = action.lower()
    if any(kw in text for kw in HIGH_RISK_KEYWORDS):
        return "high"
    if any(kw in text for kw in LOW_RISK_KEYWORDS):
        return "low"
    if any(kw in text for kw in MEDIUM_RISK_KEYWORDS):
        return "medium"
    # Explicit fail-safe default: unrecognized actions route to batched review.
    return "medium"


def tiered_gate(action: str, attempt: int = 0) -> dict:
    """Classify then gate. Low and medium tiers auto-approve; high blocks."""
    tier = classify_risk(action)
    if tier in AUTO_APPROVE_REASONS:
        return {
            "decision": "approve",
            "reason": AUTO_APPROVE_REASONS[tier],
            "action": action,
            "risk_tier": tier,
            "ts": datetime.now(timezone.utc).isoformat(),
        }
    return gate_action(action, tier, attempt=attempt)


## Модел 3: Одитен журнал и цикъл на преглед

`print("Response approved.")` не е одитен журнал. За доверие, всяко решение на вратата трябва да бъде записано като структурирано събитие, което по-късно можете да заявявате, възпроизвеждате или прикачвате към преглед на инцидент.

Два елемента:

1. **Само добавящ се JSONL.** Един ред за решение, с времева марка, действие, ниво, решение, причина. Лесно за търсене, лесно за изпращане към реално хранилище на журнали по-късно.
2. **Цикъл на преразглеждане при отказ.** Когато вратата върне `deny`, агентът се подканва отново със същия контекст на причина за отказ, така че следващото предложение да може да избегне проблема.


In [ ]:
def log_decision(decision: dict) -> None:
    """Append a gate decision to the JSONL audit log."""
    with GATE_LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(decision) + "\n")


def propose_action(goal: str, prior_rejection: str | None = None) -> str:
    """Ask the LLM to propose a concrete next action for a goal.

    If prior_rejection is provided, it is fed back so the LLM can avoid
    the same failure mode in the next proposal.
    """
    system = (
        "You are an action planner for an agent. Propose ONE concrete next\n"
        "action (a single sentence) toward the user's goal. If a prior\n"
        "rejection reason is given, propose a different action that addresses\n"
        "the rejection."
    )
    user_text = f"Goal: {goal}"
    if prior_rejection:
        user_text += f"\n\nPrior proposal was denied. Reason: {prior_rejection}"

    response = client.responses.create(
        model=deployment,
        input=[
            {"role": "system", "content": system},
            {"role": "user", "content": user_text},
        ],
        store=False,
    )
    return response.output_text.strip()


def run_with_revision(goal: str, max_revisions: int = 2) -> dict:
    """Propose, gate, and on rejection revise up to max_revisions times."""
    prior_reason: str | None = None
    for attempt in range(max_revisions + 1):
        action = propose_action(goal, prior_rejection=prior_reason)
        decision = tiered_gate(action, attempt=attempt)
        decision["attempt"] = attempt
        log_decision(decision)
        if decision["decision"] == "approve":
            return decision
        prior_reason = decision["reason"]
    return {**decision, "final": "max_revisions_reached"}



In [ ]:
# End-to-end demo: three goals at three different risk profiles.
# GATE_LOG_PATH is per-run (timestamped) so no prior log is touched.

goals = [
    "Look up the weather in Seattle for the customer's trip planning.",
    "Schedule a reminder for the customer to check in 24 hours before their flight.",
    "Send a marketing email to the customer about premium upgrade options.",
]

for goal in goals:
    print(f"\n=== Goal: {goal} ===")
    outcome = run_with_revision(goal, max_revisions=1)
    print(f"[final] {outcome['decision']} ({outcome['reason']})")

print(f"\n=== Audit log ({GATE_LOG_PATH.name}) ===")
for line in GATE_LOG_PATH.read_text(encoding="utf-8").splitlines():
    record = json.loads(line)
    print(f"  [{record['risk_tier']:6s}] {record['decision']:8s} "
          f"attempt={record.get('attempt', '?')} action={record['action'][:140]}")


## Допълнителни ресурси

Няколко други публични проекта прилагат вариации на тези HITL модели. Сравнете подходите, за да намерите този, който пасва на вашия стек:

- **LangChain** обвивки за инструменти с участие на човек ([документация](https://python.langchain.com/docs/integrations/tools/human_tools)): готови обвивки за инструменти, които спират изпълнението за човешки вход.
- **AutoGen** `UserProxyAgent` ([v0.2 документация](https://microsoft.github.io/autogen/0.2/docs/topics/human-in-the-loop); AutoGen v0.4+ го пренареди): използва агентска роля специално, за да представи човека в multi-agent разговори.
- **Microsoft Agent Framework (MAF)** middleware за извикване на функции ([документация](https://learn.microsoft.com/agent-framework/)): middleware, което се изпълнява около всяко извикване на инструмент/функция, подходящо за контрол на логика и потоци за одобрение.

Всеки проект обработва трите под-модела по различен начин: LangChain ги обвива като инструменти, AutoGen използва агентска роля, а Microsoft Agent Framework използва middleware при извикване на функции. Прочетете една или две имплементации изцяло, преди да изберете дизайн за собствения си агент.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Отказ от отговорност**:
Този документ е преведен с помощта на AI преводачески услуга [Co-op Translator](https://github.com/Azure/co-op-translator). Въпреки че се стремим към точност, моля имайте предвид, че автоматизираните преводи могат да съдържат грешки или неточности. Оригиналният документ на неговия роден език трябва да се счита за авторитетен източник. За критична информация се препоръчва професионален човешки превод. Ние не носим отговорност за каквито и да е недоразумения или неправилни тълкувания, произтичащи от използването на този превод.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
